# TFMA Analysis — COMP315 Group 5

This notebook loads the latest TFMA Evaluator artifact and reports overall model quality, performance by meaningful population slices, and fairness gaps. Every visualization is followed by a short interpretation.

In [6]:
from pathlib import Path

import pandas as pd
import tensorflow_model_analysis as tfma
from IPython.display import Markdown, display

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

EVALUATOR_ROOT = (
    PROJECT_DIR / 'pipeline_output' / 'full_pipeline' /
    'Evaluator' / 'evaluation'
)
evaluation_paths = [path for path in EVALUATOR_ROOT.iterdir() if path.is_dir()]
if not evaluation_paths:
    raise FileNotFoundError('No Evaluator artifacts found under {}'.format(EVALUATOR_ROOT))

EVALUATION_PATH = max(evaluation_paths, key=lambda path: int(path.name))
eval_result = tfma.load_eval_result(str(EVALUATION_PATH))
print('Loaded TFMA artifact:', EVALUATION_PATH)
print('Overall and feature-value slices:', len(eval_result.slicing_metrics))

Loaded TFMA artifact: /home/msingh/COMP315_Project_Group_5/pipeline_output/full_pipeline/Evaluator/evaluation/45
Overall and feature-value slices: 20


In [7]:
rows = []
for slice_key, metrics in eval_result.slicing_metrics:
    metric_values = metrics['']['']
    if slice_key:
        feature, value = slice_key[0]
        slice_name = '{}={}'.format(feature, value)
    else:
        feature, value, slice_name = 'overall', 'all clients', 'overall'
    rows.append({
        'feature': feature,
        'value': value,
        'slice': slice_name,
        'BinaryAccuracy': metric_values['binary_accuracy']['doubleValue'],
        'AUC': metric_values['auc']['doubleValue'],
    })

slice_metrics = pd.DataFrame(rows).sort_values(['feature', 'value']).reset_index(drop=True)
overall = slice_metrics[slice_metrics['feature'] == 'overall'].iloc[0]

## 1. Overall model metrics

Binary Accuracy measures correct decisions at the model's fixed threshold. AUC measures how well the model ranks positive examples across all thresholds.

In [8]:
overall_chart = pd.DataFrame({
    'metric': ['Binary Accuracy', 'AUC'],
    'score': [overall['BinaryAccuracy'], overall['AUC']],
}).set_index('metric')

display(
    overall_chart.style
    .format({'score': '{:.4f}'})
    .bar(subset=['score'], vmin=0, vmax=1, color='#4c78a8')
    .set_caption('Overall TFMA metrics')
)

,score
metric,
Binary Accuracy,0.8877
AUC,0.8422


In [9]:
display(Markdown(
    '**Interpretation.** The model achieved **{:.4f} Binary Accuracy** and '
    '**{:.4f} AUC** on the complete evaluation set. The accuracy is strong, '
    'while the AUC shows useful ranking ability. These overall values can hide '
    'weaker performance for particular groups, so the slices below are also required.'
    .format(overall['BinaryAccuracy'], overall['AUC'])
))

**Interpretation.** The model achieved **0.8877 Binary Accuracy** and **0.8422 AUC** on the complete evaluation set. The accuracy is strong, while the AUC shows useful ranking ability. These overall values can hide weaker performance for particular groups, so the slices below are also required.

## 2. Per-slice performance

The slices were selected for practical fairness reasons: `marital` is a personal demographic characteristic, while `education` and `job` may reflect socioeconomic opportunity. Comparing these groups helps reveal performance differences hidden by overall averages. The interactive TFMA view and persistent table below report every available slice.

In [10]:
tfma.view.render_slicing_metrics(eval_result)

display(
    slice_metrics[['slice', 'BinaryAccuracy', 'AUC']].set_index('slice').style
    .format({'BinaryAccuracy': '{:.4f}', 'AUC': '{:.4f}'})
    .bar(subset=['BinaryAccuracy'], vmin=0, vmax=1, color='#59a14f')
    .bar(subset=['AUC'], vmin=0, vmax=1, color='#f28e2b')
    .set_caption('Overall and per-slice TFMA performance')
)

,BinaryAccuracy,AUC
slice,,
education=primary,0.9121,0.8642
education=secondary,0.9012,0.8498
education=tertiary,0.8538,0.8251
education=unknown,0.8727,0.7991
job=admin.,0.8929,0.8667
job=blue-collar,0.9310,0.8962
job=entrepreneur,0.9137,0.8654
job=housemaid,0.8975,0.8716
job=management,0.8800,0.8563


In [11]:
summary_lines = []
for feature in ['marital', 'education', 'job']:
    group = slice_metrics[slice_metrics['feature'] == feature]
    lowest_accuracy = group.loc[group['BinaryAccuracy'].idxmin()]
    highest_accuracy = group.loc[group['BinaryAccuracy'].idxmax()]
    lowest_auc = group.loc[group['AUC'].idxmin()]
    highest_auc = group.loc[group['AUC'].idxmax()]
    summary_lines.append(
        '- **{}:** accuracy ranges from **{:.4f}** ({}) to **{:.4f}** ({}); '
        'AUC ranges from **{:.4f}** ({}) to **{:.4f}** ({}).'.format(
            feature.title(),
            lowest_accuracy['BinaryAccuracy'], lowest_accuracy['value'],
            highest_accuracy['BinaryAccuracy'], highest_accuracy['value'],
            lowest_auc['AUC'], lowest_auc['value'],
            highest_auc['AUC'], highest_auc['value'],
        )
    )

display(Markdown(
    '**Interpretation.** Performance is not uniform across the selected groups.\n\n' +
    '\n'.join(summary_lines) +
    '\n\nThe widest differences should be investigated first. A lower score means the '
    'model is less reliable for that population, but it does not by itself prove '
    'discrimination because group size and positive-label rates can also affect the result.'
))

**Interpretation.** Performance is not uniform across the selected groups.

- **Marital:** accuracy ranges from **0.8673** (single) to **0.8961** (married); AUC ranges from **0.8279** (single) to **0.8665** (divorced).
- **Education:** accuracy ranges from **0.8538** (tertiary) to **0.9121** (primary); AUC ranges from **0.7991** (unknown) to **0.8642** (primary).
- **Job:** accuracy ranges from **0.7303** (student) to **0.9310** (blue-collar); AUC ranges from **0.7778** (student) to **0.8962** (blue-collar).

The widest differences should be investigated first. A lower score means the model is less reliable for that population, but it does not by itself prove discrimination because group size and positive-label rates can also affect the result.

## 3. Fairness indicator chart

This chart shows each slice's signed difference from the overall population. A negative bar means that the slice performs below the overall result; larger absolute gaps deserve closer review. It is a diagnostic fairness indicator, not proof of unfair treatment.

In [12]:
fairness_gaps = slice_metrics[slice_metrics['feature'] != 'overall'].copy()
fairness_gaps['Accuracy gap'] = fairness_gaps['BinaryAccuracy'] - overall['BinaryAccuracy']
fairness_gaps['AUC gap'] = fairness_gaps['AUC'] - overall['AUC']
fairness_gaps = fairness_gaps.sort_values('Accuracy gap')

display(
    fairness_gaps[['slice', 'Accuracy gap', 'AUC gap']].set_index('slice').style
    .format({'Accuracy gap': '{:+.4f}', 'AUC gap': '{:+.4f}'})
    .bar(
        subset=['Accuracy gap', 'AUC gap'],
        align='mid',
        color=['#d65f5f', '#5fba7d'],
    )
    .set_caption('Fairness indicators: performance gap from overall')
)

,Accuracy gap,AUC gap
slice,,
job=student,-0.1574,-0.0644
job=retired,-0.1377,-0.0277
job=unknown,-0.0675,-0.0173
job=unemployed,-0.0544,+0.0034
education=tertiary,-0.0339,-0.0171
marital=single,-0.0204,-0.0143
education=unknown,-0.0150,-0.0431
job=self-employed,-0.0127,+0.0344
job=management,-0.0077,+0.0141


In [13]:
lowest_accuracy_gap = fairness_gaps.loc[fairness_gaps['Accuracy gap'].idxmin()]
lowest_auc_gap = fairness_gaps.loc[fairness_gaps['AUC gap'].idxmin()]

display(Markdown(
    '**Interpretation.** The largest negative accuracy gap is **{} ({:+.4f})**, '
    'and the largest negative AUC gap is **{} ({:+.4f})**. If this model were '
    'used to select clients for marketing or financial opportunities, less reliable '
    'performance could cause some groups to receive inappropriate offers or miss '
    'relevant opportunities. Before deployment, the team should examine slice counts, '
    'label rates, false-positive rates, and false-negative rates. The current Accuracy '
    'and AUC gaps identify a risk for investigation but are not enough to conclude that '
    'the model is fair or unfair.'
    .format(
        lowest_accuracy_gap['slice'], lowest_accuracy_gap['Accuracy gap'],
        lowest_auc_gap['slice'], lowest_auc_gap['AUC gap'],
    )
))

**Interpretation.** The largest negative accuracy gap is **job=student (-0.1574)**, and the largest negative AUC gap is **job=student (-0.0644)**. If this model were used to select clients for marketing or financial opportunities, less reliable performance could cause some groups to receive inappropriate offers or miss relevant opportunities. Before deployment, the team should examine slice counts, label rates, false-positive rates, and false-negative rates. The current Accuracy and AUC gaps identify a risk for investigation but are not enough to conclude that the model is fair or unfair.

## Conclusion

The model's overall results are useful, but the slice and fairness-gap views show why aggregate metrics are insufficient. `marital`, `education`, and `job` are meaningful first checks because they represent demographic or socioeconomic conditions. A production fairness review should also add slice sizes and threshold-based error metrics, and should monitor these gaps after every new model run.